In [1]:
%load_ext autoreload
%autoreload 2

In [30]:
import numpy as np
import pandas as pd
import re
import torch
from multiprocessing import Pool

from tqdm import tqdm
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset as HFDataset, load_dataset, load_from_disk, concatenate_datasets
from transformers import (
    RobertaTokenizer, 
    RobertaModel,
    RobertaForMaskedLM,
    BatchEncoding
)

from embeddings.embed_stage2 import (
    fetch_all_postings_text, 
    fetch_all_resumes_text,
    doc_sim_score
)

from embeddings.contrastive_learning import (
    AUGMENTATION_FNS,
    ContrastiveLearningDataset, 
    ContrastiveLearningModel,
    soft_alignment_loss,
    prepare_token_dataset,
    train_model
)

In [6]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaModel.from_pretrained('./roberta-tuned-v1', add_pooling_layer=False, output_hidden_states=True)

In [7]:
BLOCK_SIZE = 128 # Stride length when splitting long texts into 512-length segments
MAX_SEQ_LEN = 512 # maximum length of a sequence that BERT can operate on

posting_txt_col = 'description'
resume_txt_col = 'Resume_str'

DEVICE = (f'cuda:0' if torch.cuda.is_available() else 'cpu')

In [9]:
posting_db_url = 'sqlite:///postings.db'
resume_db_url = 'sqlite:///resumes.db'
postings = fetch_all_postings_text(posting_db_url)
resumes = fetch_all_resumes_text(resume_db_url)

In [10]:
# postings[0]

In [11]:
# split_sentences(resumes[4])

In [12]:
# lm_dataset = prepare_token_dataset(tokenizer, posting_path='./linkedin_data/sample_postings.csv', resume_path='./resume_data/sample_resumes.csv')
# lm_dataset = prepare_token_dataset(tokenizer, posting_path='./temp/postings15k.csv', resume_path='./resume_data/Resume.csv')

# posting_df = pd.read_csv('/home/hice1/khom9/scratch/CS6220_Project/postings.csv').dropna(subset=['description'])
# lm_dataset = prepare_token_dataset(tokenizer, posting_df=posting_df, resume_path='./resume_data/Resume.csv')
lm_dataset = load_from_disk('/home/hice1/khom9/scratch/CS6220_Project/lm_dataset')

In [13]:
# lm_dataset.save_to_disk('/home/hice1/khom9/scratch/CS6220_Project/lm_dataset')

In [15]:
# lm_dataset.set_format('torch')
d = ContrastiveLearningDataset(lm_dataset, 'train', tokenizer, AUGMENTATION_FNS, num_sample=5000)

In [16]:
lm_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 369363
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 92095
    })
})

In [24]:
batch_size = 16
lr = 1e-5
m = ContrastiveLearningModel(model, out_embed_dim=588).to(DEVICE)
optimizer = optim.Adam(m.parameters(), lr=lr)
loss_fn = soft_alignment_loss #nn.TripletMarginLoss()
save_path = './temp/contrastive_learning.pth'

epochs = 10

train_model(m, optimizer, d, loss_fn, epochs, batch_size, device=DEVICE, save_path=None, save_freq=1)


In [ ]:
from embed_stage2 import fetch_all_postings_text, fetch_all_resumes_text, load_model

posting_db_url = 'sqlite:///postings.db'
resume_db_url = 'sqlite:///resumes.db'
postings = fetch_all_postings_text(posting_db_url)
resumes = fetch_all_resumes_text(resume_db_url)

# mm = ContrastiveLearningModel(model).to(DEVICE)
# mm = load_model(mm, save_path)

In [ ]:
t = doc_sim_score(postings[2], resumes[4], DEVICE)
t.cpu().item()

In [ ]:
# loader = DataLoader(d, batch_size=32, shuffle=False)
# y = next(iter(loader))

# mm({k: v.to(DEVICE) for k,v in y.items()})